# Wikidata ORCID Author Reconciliation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_ORCID_Author_Reconciliation.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook converts author name strings (P2093) to linked author items (P50) in Wikidata for scholarly articles, using **ORCID identifiers as the sole matching criterion**. Unlike general author reconciliation approaches that rely on fuzzy name matching, this notebook requires explicit ORCID matches to ensure high-confidence linkage.

The workflow queries CrossRef to retrieve author metadata including affiliation at time of publication, then generates QuickStatements that add rich P50 statements with:
- **Series ordinal (P1545)**: Author position in byline
- **Object named as (P1932)**: Author name as it appeared in publication
- **Affiliation string (P6424)**: Author's institutional affiliation
- **References**: Crossref as stated source with reference URL

This approach trades recall for precision — only authors with verified ORCID matches are processed, but those matches are reliable and richly documented.

## Key Features

- **ORCID-Only Matching**: No fuzzy name matching — only processes authors with verifiable ORCID links
- **CrossRef Integration**: Fetches author affiliations and name-as-published from CrossRef API
- **Rich QuickStatements**: Generates P50 statements with full qualifier set and references
- **Automatic P2093 Removal**: Creates removal commands using exact Wikidata string values
- **Scholarly Endpoint Aware**: Uses query-scholarly.wikidata.org for article lookups
- **Batch ORCID Lookup**: Resolves multiple ORCIDs in a single SPARQL query for efficiency

## Workflow

1. **Upload**: CrossRef-sourced article CSV with Authors, DOI, and ORCIDs columns
2. **Resolve ORCIDs**: Batch lookup ORCIDs against Wikidata person items
3. **Fetch Metadata**: Get author affiliations and names from CrossRef API
4. **Match**: Align ORCID-identified authors to existing P2093 statements
5. **Generate**: Create QuickStatements to add P50 (with qualifiers/references) and remove P2093
6. **Export**: Download batch file for upload to QuickStatements

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
!pip install requests pandas ipywidgets rapidfuzz -q

import requests
import pandas as pd
import re
import time
import json
from datetime import datetime
from collections import defaultdict
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO
from rapidfuzz import fuzz

print("Setup complete.")

## Configuration

*Define endpoints, constants, and styling.*

In [ ]:
# Wikidata endpoints
# Since May 2025, scholarly articles are ONLY on the scholarly endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"
MAIN_ENDPOINT = "https://query.wikidata.org/sparql"

# CrossRef API
CROSSREF_WORKS_URL = "https://api.crossref.org/works/"

# Wikidata Q-IDs for references
CROSSREF_QID = "Q5188229"  # Crossref organization

# Wikidata Property IDs
P50 = "P50"      # author
P2093 = "P2093"  # author name string
P1545 = "P1545"  # series ordinal
P1932 = "P1932"  # object named as
P6424 = "P6424"  # affiliation string
P248 = "S248"    # stated in (as source)
P854 = "S854"    # reference URL (as source)

# User agent for API requests
USER_AGENT = "WikidataORCIDReconciliation/1.0 (matt@mattartz.me; Wikidata bot)"

# Color palette
COLORS = {
    'bg_primary': '#E7ECEF',
    'text_primary': '#274C77',
    'interactive': '#6096BA',
    'bg_secondary': '#A3CEF1',
    'neutral': '#8B8C89',
    'success': '#28a745',
    'warning': '#ffc107'
}

# Styling for containers
CONTAINER_STYLE = f"""
    background-color: {COLORS['bg_primary']};
    border-left: 5px solid {COLORS['text_primary']};
    border-radius: 10px;
    padding: 15px;
    margin: 10px 0;
"""

print(f"Scholarly endpoint: {SCHOLARLY_ENDPOINT}")
print(f"Main endpoint: {MAIN_ENDPOINT}")
print(f"CrossRef reference QID: {CROSSREF_QID}")

## Helper Functions: SPARQL Queries

*Functions to query Wikidata for person items by ORCID and article P2093 data.*

In [ ]:
def sparql_query(endpoint, query, timeout=30):
    """Execute a SPARQL query and return results."""
    try:
        response = requests.get(
            endpoint,
            params={"query": query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=timeout
        )
        response.raise_for_status()
        return response.json().get("results", {}).get("bindings", [])
    except requests.exceptions.Timeout:
        print(f"   Timeout querying {endpoint}")
        return []
    except requests.exceptions.RequestException as e:
        print(f"   Request error: {e}")
        return []
    except Exception as e:
        print(f"   Unexpected error: {e}")
        return []


def clean_orcid(orcid_input):
    """
    Extract clean ORCID from various formats.
    Returns 16-digit ORCID or None.
    """
    if not orcid_input or not isinstance(orcid_input, str):
        return None
    
    orcid_str = str(orcid_input).strip()
    
    # Extract from URL format
    if 'orcid.org/' in orcid_str:
        orcid_str = orcid_str.split('orcid.org/')[-1].strip(']').strip()
    
    # Validate format (0000-0000-0000-000X)
    orcid_pattern = r'^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$'
    if re.match(orcid_pattern, orcid_str):
        return orcid_str
    
    return None


def lookup_person_by_orcid(orcid):
    """
    Look up a person in Wikidata by ORCID.
    Returns dict with QID and basic info, or None if not found.
    """
    orcid_clean = clean_orcid(orcid)
    if not orcid_clean:
        return None

    query = f"""
    SELECT ?person ?personLabel WHERE {{
      ?person wdt:P496 "{orcid_clean}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """

    results = sparql_query(MAIN_ENDPOINT, query)

    if results:
        r = results[0]
        qid = r.get('person', {}).get('value', '').split('/')[-1]
        return {
            'qid': qid,
            'label': r.get('personLabel', {}).get('value', ''),
            'orcid': orcid_clean
        }
    return None


def batch_lookup_orcids(orcids):
    """
    Look up multiple ORCIDs in a single SPARQL query.
    Returns dict mapping ORCID -> {qid, label}
    """
    # Clean and filter valid ORCIDs
    valid_orcids = [clean_orcid(o) for o in orcids if clean_orcid(o)]
    if not valid_orcids:
        return {}
    
    # Batch in groups of 50 to avoid query limits
    results_map = {}
    batch_size = 50
    
    for i in range(0, len(valid_orcids), batch_size):
        batch = valid_orcids[i:i+batch_size]
        orcid_values = ' '.join([f'"{o}"' for o in batch])
        
        query = f"""
        SELECT ?person ?personLabel ?orcid WHERE {{
          VALUES ?orcid {{ {orcid_values} }}
          ?person wdt:P496 ?orcid .
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """
        
        results = sparql_query(MAIN_ENDPOINT, query, timeout=60)
        
        for r in results:
            orcid = r.get('orcid', {}).get('value', '')
            qid = r.get('person', {}).get('value', '').split('/')[-1]
            label = r.get('personLabel', {}).get('value', '')
            results_map[orcid] = {'qid': qid, 'label': label}
        
        time.sleep(0.5)  # Rate limiting between batches
    
    return results_map


print("SPARQL query functions loaded.")

## Helper Functions: CrossRef API

*Functions to fetch author metadata from CrossRef including affiliations and name as published.*

In [ ]:
def clean_doi(doi_input):
    """Extract clean DOI from various formats."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    doi_input = str(doi_input).strip()
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()
    return None


def get_crossref_author_data(doi):
    """
    Fetch author data from CrossRef API for a given DOI.
    Returns list of authors with: name, orcid, affiliation, ordinal.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None
    
    url = f"{CROSSREF_WORKS_URL}{doi_clean}"
    
    try:
        response = requests.get(
            url,
            headers={"User-Agent": USER_AGENT},
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return None
    
    work = data.get('message', {})
    authors_raw = work.get('author', [])
    
    authors = []
    for idx, author in enumerate(authors_raw, 1):
        # Build full name
        given = author.get('given', '')
        family = author.get('family', '')
        full_name = f"{given} {family}".strip() if given else family
        
        # Extract ORCID if present
        orcid = None
        if author.get('ORCID'):
            orcid = clean_orcid(author['ORCID'])
        
        # Extract affiliation(s)
        affiliations = []
        for aff in author.get('affiliation', []):
            if aff.get('name'):
                affiliations.append(aff['name'])
        affiliation_str = '; '.join(affiliations) if affiliations else None
        
        authors.append({
            'name': full_name,
            'given': given,
            'family': family,
            'orcid': orcid,
            'affiliation': affiliation_str,
            'ordinal': str(idx)
        })
    
    return {
        'doi': doi_clean,
        'crossref_url': f"https://api.crossref.org/v1/works/{doi_clean}",
        'authors': authors
    }


print("CrossRef API functions loaded.")

## Helper Functions: Article Lookup

*Functions to find articles in Wikidata and retrieve existing P2093 author name strings.*

In [ ]:
def lookup_article_with_authors(doi):
    """
    Look up an article in the scholarly endpoint by DOI.
    Returns article QID and all P2093 author name strings with their ordinals.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None
    
    doi_upper = doi_clean.upper()

    query = f"""
    SELECT ?article ?articleLabel ?authorNameString ?ordinal WHERE {{
      ?article wdt:P356 "{doi_upper}" .
      OPTIONAL {{
        ?article p:P2093 ?authorStmt .
        ?authorStmt ps:P2093 ?authorNameString .
        OPTIONAL {{ ?authorStmt pq:P1545 ?ordinal }}
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    results = sparql_query(SCHOLARLY_ENDPOINT, query)

    if not results:
        return None

    article_qid = results[0].get('article', {}).get('value', '').split('/')[-1]
    article_label = results[0].get('articleLabel', {}).get('value', '')

    # Collect all P2093 author name strings with ordinals
    author_strings = {}
    for r in results:
        name = r.get('authorNameString', {}).get('value', '')
        if name:
            ordinal = r.get('ordinal', {}).get('value', '')
            author_strings[name] = {
                'exact_name': name,
                'ordinal': ordinal
            }

    return {
        'qid': article_qid,
        'label': article_label,
        'author_strings': author_strings
    }


def match_author_to_p2093(author_name, wikidata_author_strings):
    """
    Match a CrossRef author name to the best matching P2093 value in Wikidata.
    Uses fuzzy matching to handle minor differences.
    Returns dict with exact_name and ordinal, or None if no good match.
    """
    if not wikidata_author_strings or not author_name:
        return None

    name_lower = author_name.lower().strip()
    best_match = None
    best_score = 0

    for wd_name, wd_data in wikidata_author_strings.items():
        # Calculate similarity
        score = fuzz.ratio(name_lower, wd_name.lower())
        # Also try token sort ratio for name order differences
        token_score = fuzz.token_sort_ratio(name_lower, wd_name.lower())
        score = max(score, token_score)

        if score > best_score:
            best_score = score
            best_match = wd_data

    # Require at least 75% match
    if best_score >= 75:
        return best_match
    return None


print("Article lookup functions loaded.")

## Helper Functions: Data Parsing

*Functions to parse author names and ORCIDs from CSV data.*

In [ ]:
def parse_orcids_from_string(orcids_string):
    """
    Parse ORCIDs from the ORCIDs column format.
    Format: "Surname1 [https://orcid.org/0000-0000-0000-0000], Surname2 [https://orcid.org/...]"
    Returns dict mapping surname/name part -> ORCID
    """
    orcid_map = {}
    if not orcids_string or pd.isna(orcids_string):
        return orcid_map
    
    # Pattern: Name [https://orcid.org/XXXX-XXXX-XXXX-XXXX]
    pattern = r'([^\[,]+)\s*\[https?://orcid\.org/([^\]]+)\]'
    matches = re.findall(pattern, str(orcids_string))
    
    for name_part, orcid in matches:
        name_part = name_part.strip().rstrip(',')
        orcid_clean = clean_orcid(orcid)
        if orcid_clean:
            orcid_map[name_part.lower()] = orcid_clean
    
    return orcid_map


def extract_dois_with_orcids(df):
    """
    Extract DOIs that have at least one ORCID in their author list.
    Returns list of DOIs to process.
    """
    dois = []
    
    for idx, row in df.iterrows():
        doi = clean_doi(row.get('DOI', ''))
        orcids = row.get('ORCIDs', '')
        
        if doi and orcids and not pd.isna(orcids):
            orcid_map = parse_orcids_from_string(orcids)
            if orcid_map:  # Has at least one ORCID
                dois.append(doi)
    
    return list(set(dois))  # Deduplicate


print("Data parsing functions loaded.")

## Test Endpoint Connections

*Verify both Wikidata endpoints and CrossRef API are accessible.*

In [ ]:
print("Testing connections...")
print()

# Test main endpoint with ORCID lookup
print("1. Main endpoint (ORCID lookup):")
test_orcid = "0000-0003-3789-2359"  # Example ORCID
result = lookup_person_by_orcid(test_orcid)
if result:
    print(f"   ✓ Found: {result['label']} ({result['qid']})")
else:
    print(f"   No result for ORCID {test_orcid}")

print()

# Test scholarly endpoint
print("2. Scholarly endpoint (article lookup):")
test_doi = "10.1111/napa.70016"
result = lookup_article_with_authors(test_doi)
if result:
    print(f"   ✓ Found: {result['qid']}")
    print(f"   Title: {result['label'][:60]}..." if len(result['label']) > 60 else f"   Title: {result['label']}")
    print(f"   P2093 count: {len(result['author_strings'])}")
else:
    print(f"   No result for DOI {test_doi}")

print()

# Test CrossRef API
print("3. CrossRef API (author metadata):")
cr_result = get_crossref_author_data(test_doi)
if cr_result:
    print(f"   ✓ Found {len(cr_result['authors'])} authors")
    for a in cr_result['authors'][:2]:
        orcid_str = f" [ORCID: {a['orcid']}]" if a['orcid'] else ""
        aff_str = f" @ {a['affiliation'][:40]}..." if a.get('affiliation') else ""
        print(f"      - {a['name']}{orcid_str}{aff_str}")
else:
    print(f"   No CrossRef data for {test_doi}")

print()
print("Connection tests complete.")

## Upload Article Data

*Upload a CSV file with article metadata. Required columns: DOI, ORCIDs.*

In [ ]:
articles_df = None
dois_with_orcids = []

# File upload widget
file_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV'
)

upload_output = widgets.Output()

def on_upload(change):
    global articles_df, dois_with_orcids
    with upload_output:
        clear_output()
        if file_upload.value:
            try:
                file_info = list(file_upload.value.values())[0]
                articles_df = pd.read_csv(BytesIO(file_info['content']))

                # Filter to only articles if Type column exists
                if 'Type' in articles_df.columns:
                    articles_df = articles_df[articles_df['Type'] == 'journal-article'].copy()

                # Extract DOIs that have ORCIDs
                dois_with_orcids = extract_dois_with_orcids(articles_df)

                print(f"Loaded {len(articles_df)} articles")
                print(f"Articles with at least one ORCID: {len(dois_with_orcids)}")
                print()

                # Count total ORCIDs
                total_orcids = 0
                for _, row in articles_df.iterrows():
                    orcid_map = parse_orcids_from_string(row.get('ORCIDs', ''))
                    total_orcids += len(orcid_map)
                
                print(f"Total ORCID instances in data: {total_orcids}")
                print()
                
                # Preview
                print("Sample DOIs with ORCIDs:")
                for doi in dois_with_orcids[:5]:
                    print(f"  {doi}")

            except Exception as e:
                print(f"Error loading file: {e}")

file_upload.observe(on_upload, names='value')

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📁 Upload Article Data</h3>
    <p>Upload a CSV file with columns: DOI, ORCIDs.</p>
    <p><em>Only articles with at least one ORCID will be processed.</em></p>
</div>
"""))
display(file_upload)
display(upload_output)

## Process ORCID Matches

*For each DOI with ORCIDs, look up person Q-IDs in Wikidata and fetch author metadata from CrossRef.*

In [ ]:
# Store processing results
matched_authors = {}  # doi -> list of matched author data
orcid_to_qid = {}     # orcid -> qid (cache)
article_cache = {}    # doi -> wikidata article info
crossref_cache = {}   # doi -> crossref author data

# Configuration widgets
process_button = widgets.Button(
    description='Process ORCID Matches',
    button_style='primary',
    icon='cogs'
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

process_output = widgets.Output()

def run_processing(button):
    global matched_authors, orcid_to_qid, article_cache, crossref_cache
    matched_authors = {}
    
    with process_output:
        clear_output()
        
        if not dois_with_orcids:
            print("Please upload article data first.")
            return
        
        total = len(dois_with_orcids)
        progress_bar.max = total
        progress_bar.value = 0
        
        print(f"Processing {total} articles with ORCIDs...")
        print("=" * 60)
        print()
        
        # First, collect all unique ORCIDs for batch lookup
        print("Step 1: Collecting unique ORCIDs...")
        all_orcids = set()
        for _, row in articles_df.iterrows():
            orcid_map = parse_orcids_from_string(row.get('ORCIDs', ''))
            all_orcids.update(orcid_map.values())
        
        print(f"   Found {len(all_orcids)} unique ORCIDs")
        
        # Batch lookup ORCIDs in Wikidata
        print("Step 2: Looking up ORCIDs in Wikidata...")
        orcid_to_qid = batch_lookup_orcids(list(all_orcids))
        print(f"   {len(orcid_to_qid)} ORCIDs found in Wikidata")
        print(f"   {len(all_orcids) - len(orcid_to_qid)} ORCIDs NOT in Wikidata")
        print()
        
        # Process each article
        print("Step 3: Processing articles...")
        articles_processed = 0
        authors_matched = 0
        no_wikidata = 0
        no_crossref = 0
        
        for idx, doi in enumerate(dois_with_orcids):
            progress_bar.value = idx + 1
            
            # Get the row for this DOI
            row = articles_df[articles_df['DOI'].apply(clean_doi) == doi].iloc[0]
            csv_orcids = parse_orcids_from_string(row.get('ORCIDs', ''))
            
            # Check if article exists in Wikidata
            if doi not in article_cache:
                article_info = lookup_article_with_authors(doi)
                article_cache[doi] = article_info
                time.sleep(0.3)
            else:
                article_info = article_cache[doi]
            
            if not article_info:
                no_wikidata += 1
                continue
            
            # Get CrossRef author data
            if doi not in crossref_cache:
                crossref_data = get_crossref_author_data(doi)
                crossref_cache[doi] = crossref_data
                time.sleep(0.3)
            else:
                crossref_data = crossref_cache[doi]
            
            if not crossref_data:
                no_crossref += 1
                continue
            
            # Match authors by ORCID
            article_matches = []
            
            for cr_author in crossref_data['authors']:
                if not cr_author['orcid']:
                    continue
                
                # Check if this ORCID is in Wikidata
                if cr_author['orcid'] not in orcid_to_qid:
                    continue
                
                person_qid = orcid_to_qid[cr_author['orcid']]['qid']
                
                # Find matching P2093 in Wikidata
                p2093_match = match_author_to_p2093(
                    cr_author['name'],
                    article_info['author_strings']
                )
                
                if p2093_match:
                    article_matches.append({
                        'person_qid': person_qid,
                        'orcid': cr_author['orcid'],
                        'name_as_published': cr_author['name'],
                        'affiliation': cr_author['affiliation'],
                        'ordinal': p2093_match['ordinal'] or cr_author['ordinal'],
                        'exact_p2093': p2093_match['exact_name'],
                        'crossref_url': crossref_data['crossref_url']
                    })
                    authors_matched += 1
            
            if article_matches:
                matched_authors[doi] = {
                    'article_qid': article_info['qid'],
                    'article_title': article_info['label'],
                    'matches': article_matches
                }
                articles_processed += 1
        
        print()
        print("=" * 60)
        print("PROCESSING SUMMARY")
        print("=" * 60)
        print(f"Articles processed: {articles_processed}")
        print(f"Total authors matched: {authors_matched}")
        print(f"Articles not in Wikidata: {no_wikidata}")
        print(f"Articles not in CrossRef: {no_crossref}")
        print()
        
        # Preview some matches
        if matched_authors:
            print("Sample matches:")
            for doi, data in list(matched_authors.items())[:3]:
                print(f"  {doi} ({data['article_qid']})")
                for m in data['matches'][:2]:
                    aff_preview = f" @ {m['affiliation'][:30]}..." if m.get('affiliation') else ""
                    print(f"    → {m['name_as_published']} ({m['person_qid']}){aff_preview}")

process_button.on_click(run_processing)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">⚙️ Process ORCID Matches</h3>
    <p>Match authors by ORCID to Wikidata person items and fetch CrossRef metadata.</p>
    <p><em>Only authors with verified ORCID matches will be included.</em></p>
</div>
"""))
display(widgets.VBox([
    process_button,
    progress_bar,
    process_output
]))

## Review Matches

*Review the matched authors before generating QuickStatements.*

In [ ]:
review_output = widgets.Output()

def show_review():
    with review_output:
        clear_output()
        
        if not matched_authors:
            print("No matches to review. Run processing first.")
            return
        
        total_matches = sum(len(data['matches']) for data in matched_authors.values())
        
        print(f"REVIEW: {total_matches} author matches across {len(matched_authors)} articles")
        print("=" * 70)
        print()
        
        for doi, data in matched_authors.items():
            print(f"📄 {data['article_title'][:60]}..." if len(data['article_title']) > 60 else f"📄 {data['article_title']}")
            print(f"   DOI: {doi}")
            print(f"   Article QID: {data['article_qid']}")
            print()
            
            for m in data['matches']:
                print(f"   👤 {m['name_as_published']}")
                print(f"      Person QID: {m['person_qid']}")
                print(f"      ORCID: {m['orcid']}")
                print(f"      Ordinal: {m['ordinal']}")
                if m.get('affiliation'):
                    print(f"      Affiliation: {m['affiliation'][:60]}..." if len(m['affiliation']) > 60 else f"      Affiliation: {m['affiliation']}")
                print(f"      P2093 to remove: \"{m['exact_p2093']}\"")
                print()
            
            print("-" * 70)
            print()

review_button = widgets.Button(
    description='Review Matches',
    button_style='info',
    icon='eye'
)
review_button.on_click(lambda b: show_review())

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">👀 Review Matches</h3>
    <p>Review all ORCID-based matches before generating QuickStatements.</p>
</div>
"""))
display(review_button)
display(review_output)

## Generate QuickStatements

*Generate QuickStatements with full qualifiers and references:*
- Add P50 (author) with series ordinal, object named as, affiliation string
- Add reference: stated in Crossref with reference URL
- Remove P2093 (author name string)

In [ ]:
def escape_qs_string(s):
    """Escape a string for QuickStatements V1 format."""
    if not s:
        return s
    # Escape double quotes by doubling them
    return s.replace('"', '""')


def generate_quickstatements(include_removals=True):
    """
    Generate QuickStatements V1 format commands.
    
    For each matched author:
    - Add P50 with qualifiers: P1545 (ordinal), P1932 (name as published), P6424 (affiliation)
    - Add sources: S248 (stated in Crossref), S854 (reference URL)
    - Remove P2093 with exact string value
    """
    qs_lines = []
    
    for doi, data in matched_authors.items():
        article_qid = data['article_qid']
        
        for m in data['matches']:
            # Build the P50 statement with qualifiers and sources
            parts = [article_qid, P50, m['person_qid']]
            
            # Add series ordinal (P1545)
            if m.get('ordinal'):
                parts.extend([P1545, f'"{m["ordinal"]}"'])
            
            # Add object named as (P1932)
            if m.get('name_as_published'):
                name_escaped = escape_qs_string(m['name_as_published'])
                parts.extend([P1932, f'"{name_escaped}"'])
            
            # Add affiliation string (P6424)
            if m.get('affiliation'):
                aff_escaped = escape_qs_string(m['affiliation'])
                parts.extend([P6424, f'"{aff_escaped}"'])
            
            # Add references: stated in Crossref (S248), reference URL (S854)
            parts.extend([P248, CROSSREF_QID])
            if m.get('crossref_url'):
                url_escaped = escape_qs_string(m['crossref_url'])
                parts.extend([P854, f'"{url_escaped}"'])
            
            qs_lines.append('|'.join(parts))
            
            # Add P2093 removal
            if include_removals and m.get('exact_p2093'):
                p2093_escaped = escape_qs_string(m['exact_p2093'])
                qs_lines.append(f'-{article_qid}|{P2093}|"{p2093_escaped}"')
    
    return '\n'.join(qs_lines)


# Widgets
include_removals_checkbox = widgets.Checkbox(
    value=True,
    description='Include P2093 removal commands',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description='Generate QuickStatements',
    button_style='success',
    icon='download'
)

generate_output = widgets.Output()

def run_generate(button):
    with generate_output:
        clear_output()
        
        if not matched_authors:
            print("No matches to generate. Run processing first.")
            return
        
        print("Generating QuickStatements...")
        print()
        
        qs_text = generate_quickstatements(
            include_removals=include_removals_checkbox.value
        )
        
        if not qs_text:
            print("No QuickStatements generated.")
            return
        
        # Save to file
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"orcid_author_reconciliation_{timestamp}.txt"
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(qs_text)
        
        line_count = len(qs_text.strip().split('\n'))
        total_matches = sum(len(data['matches']) for data in matched_authors.values())
        
        print(f"Generated {line_count} QuickStatements commands")
        print(f"Covering {total_matches} author matches across {len(matched_authors)} articles")
        print(f"Saved to: {filename}")
        print()
        
        # Preview
        print("--- PREVIEW (first 5 commands) ---")
        preview_lines = qs_text.strip().split('\n')[:5]
        for line in preview_lines:
            # Truncate long lines for display
            if len(line) > 120:
                print(line[:120] + "...")
            else:
                print(line)
        print()
        
        print("--- FORMAT EXPLANATION ---")
        print('Add P50:    ARTICLE|P50|PERSON|P1545|"ord"|P1932|"name"|P6424|"affil"|S248|Q5188229|S854|"url"')
        print('Remove P2093: -ARTICLE|P2093|"Exact Name"')
        print()
        print("Qualifiers added:")
        print("  - P1545: Series ordinal (author position)")
        print("  - P1932: Object named as (name as published)")
        print("  - P6424: Affiliation string")
        print()
        print("References added:")
        print("  - S248: Stated in Crossref (Q5188229)")
        print("  - S854: Reference URL (CrossRef API)")
        print()
        
        print("--- UPLOAD INSTRUCTIONS ---")
        print("1. Go to: https://quickstatements.toolforge.org/")
        print("2. Log in with your Wikidata account")
        print("3. Click 'New batch'")
        print("4. Paste the file contents")
        print("5. Click 'Import V1 commands'")
        print("6. Review carefully and click 'Run'")
        print()
        print("NOTE: Lines starting with '-' will REMOVE statements.")
        
        # Download in Colab
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

generate_button.on_click(run_generate)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📤 Generate QuickStatements</h3>
    <p>Generate QuickStatements with full qualifiers and Crossref references.</p>
    <p><strong>P50 includes:</strong> ordinal, name as published, affiliation, Crossref reference</p>
</div>
"""))
display(widgets.VBox([
    include_removals_checkbox,
    generate_button,
    generate_output
]))

## Export Summary

*Export a CSV summary of all ORCID matches for documentation.*

In [ ]:
export_button = widgets.Button(
    description='Export Summary CSV',
    button_style='info',
    icon='table'
)

export_output = widgets.Output()

def run_export(button):
    with export_output:
        clear_output()
        
        if not matched_authors:
            print("No data to export. Run processing first.")
            return
        
        rows = []
        for doi, data in matched_authors.items():
            for m in data['matches']:
                rows.append({
                    'DOI': doi,
                    'Article_QID': data['article_qid'],
                    'Article_Title': data['article_title'],
                    'Author_Name': m['name_as_published'],
                    'Person_QID': m['person_qid'],
                    'ORCID': m['orcid'],
                    'Ordinal': m['ordinal'],
                    'Affiliation': m.get('affiliation', ''),
                    'P2093_Value': m['exact_p2093']
                })
        
        df = pd.DataFrame(rows)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"orcid_reconciliation_summary_{timestamp}.csv"
        
        df.to_csv(filename, index=False)
        
        print(f"Exported: {filename}")
        print(f"Total matches: {len(rows)}")
        print(f"Unique articles: {len(matched_authors)}")
        print(f"Unique persons: {df['Person_QID'].nunique()}")
        
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

export_button.on_click(run_export)

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📊 Export Summary</h3>
    <p>Export a CSV summary of all ORCID-based matches for documentation.</p>
</div>
"""))
display(export_button)
display(export_output)

## Statistics: ORCID Coverage

*Analyze ORCID coverage and matching rates.*

In [ ]:
stats_button = widgets.Button(
    description='Show Statistics',
    button_style='info',
    icon='bar-chart'
)

stats_output = widgets.Output()

def show_stats(button):
    with stats_output:
        clear_output()
        
        if articles_df is None:
            print("Please upload article data first.")
            return
        
        print("ORCID COVERAGE STATISTICS")
        print("=" * 50)
        print()
        
        # Count ORCIDs in source data
        total_orcids_in_csv = 0
        unique_orcids_in_csv = set()
        
        for _, row in articles_df.iterrows():
            orcid_map = parse_orcids_from_string(row.get('ORCIDs', ''))
            total_orcids_in_csv += len(orcid_map)
            unique_orcids_in_csv.update(orcid_map.values())
        
        print(f"Source Data:")
        print(f"  Total articles: {len(articles_df)}")
        print(f"  Articles with ORCIDs: {len(dois_with_orcids)}")
        print(f"  Total ORCID instances: {total_orcids_in_csv}")
        print(f"  Unique ORCIDs: {len(unique_orcids_in_csv)}")
        print()
        
        if orcid_to_qid:
            in_wikidata = len(orcid_to_qid)
            not_in_wikidata = len(unique_orcids_in_csv) - in_wikidata
            
            print(f"Wikidata Coverage:")
            print(f"  ORCIDs found in Wikidata: {in_wikidata} ({100*in_wikidata/len(unique_orcids_in_csv):.1f}%)")
            print(f"  ORCIDs NOT in Wikidata: {not_in_wikidata} ({100*not_in_wikidata/len(unique_orcids_in_csv):.1f}%)")
            print()
        
        if matched_authors:
            total_matches = sum(len(data['matches']) for data in matched_authors.values())
            
            print(f"Matching Results:")
            print(f"  Articles with successful matches: {len(matched_authors)}")
            print(f"  Total author matches: {total_matches}")
            print()
            
            # Affiliation coverage
            with_affiliation = 0
            for data in matched_authors.values():
                for m in data['matches']:
                    if m.get('affiliation'):
                        with_affiliation += 1
            
            print(f"  Matches with affiliation: {with_affiliation} ({100*with_affiliation/total_matches:.1f}%)")

stats_button.on_click(show_stats)

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📈 Statistics</h3>
    <p>View ORCID coverage and matching statistics.</p>
</div>
"""))
display(stats_button)
display(stats_output)